# Project 4 audio inference
Run all cells with the Project 4 competition data and the modified Freesound model attached. If the Kaggle model version contains only `modified_weights.pkl`, attach its matching `model.joblib` as an additional input. The kernel classifier needs both the coefficients and the fitted acoustic reference vectors and scalers.

The notebook scores the test WAVs and writes three course codes per recording to `/kaggle/working/submission.csv`. The fixed class-name mapping comes from checked course training clips. The Freesound training pool contains confirmed matches to 1,432 of these 1,592 recordings, so use this for the competition only if that overlap is permitted. The 90% training accuracy measures performance on the fitted Freesound clips.

In [ ]:
import os
from pathlib import Path
from functools import lru_cache
from io import BytesIO
from math import gcd
from zipfile import ZipFile
import hashlib
import joblib
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.fft import dct, irfft, rfft
from scipy.ndimage import correlate1d, gaussian_filter1d, median_filter, uniform_filter1d
from scipy.signal import resample_poly
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

MODEL_DIR = Path("/kaggle/input/models/rahul24451/final-model/scikitlearn/default/1")
INPUT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working/submission.csv")
# Optionally set DATA to the extracted project4 directory or project4.zip.
DATA = None

In [ ]:
MODEL_HASH = "aad85a0fd647266b03fedbd828f0c2e184649327f3274b81fc0dc5b839760071"
WEIGHT_HASH = "7d80164c83bcde2f54fa1f54630885bf807e4f07ef2c9571e647bd43a5b80e3c"
ID_HASH = '361faf1c6e0c845ee0f62fd15787ca2b27b7fc5c599149e6d4767ee77e7562f8'
COURSE_CODES = {
    "Acoustic_guitar": "C22",
    "Applause": "C21",
    "Bark": "C30",
    "Bass_drum": "C09",
    "Burping_or_eructation": "C41",
    "Bus": "C15",
    "Cello": "C12",
    "Chime": "C36",
    "Clarinet": "C11",
    "Computer_keyboard": "C02",
    "Cough": "C17",
    "Cowbell": "C06",
    "Double_bass": "C07",
    "Drawer_open_or_close": "C29",
    "Electric_piano": "C32",
    "Fart": "C31",
    "Finger_snapping": "C14",
    "Fireworks": "C27",
    "Flute": "C26",
    "Glockenspiel": "C10",
    "Gong": "C24",
    "Gunshot_or_gunfire": "C03",
    "Harmonica": "C28",
    "Hi-hat": "C33",
    "Keys_jangling": "C08",
    "Knock": "C01",
    "Laughter": "C37",
    "Meow": "C13",
    "Microwave_oven": "C35",
    "Oboe": "C19",
    "Saxophone": "C05",
    "Scissors": "C38",
    "Shatter": "C23",
    "Snare_drum": "C18",
    "Squeak": "C34",
    "Tambourine": "C16",
    "Tearing": "C25",
    "Telephone": "C39",
    "Trumpet": "C04",
    "Violin_or_fiddle": "C40",
    "Writing": "C20"
}

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Attach the Kaggle model at {MODEL_DIR}")
artifacts = sorted(p for p in MODEL_DIR.rglob("*") if p.suffix.lower() in {".pkl", ".joblib"})
weights = None
full_model_path = None
for path in artifacts:
    digest = sha256(path)
    if digest == WEIGHT_HASH:
        weights = joblib.load(path)
    if digest == MODEL_HASH:
        full_model_path = path

if weights is None and full_model_path is None:
    raise FileNotFoundError(f"Expected the modified model.joblib or modified_weights.pkl under {MODEL_DIR}; found {artifacts}")
if full_model_path is None:
    # A second attached Kaggle input can supply the matching reference vectors and scalers.
    full_model_path = next((p for p in INPUT.rglob("model.joblib") if sha256(p) == MODEL_HASH), None)
if full_model_path is None:
    raise FileNotFoundError("The mounted PKL contains coefficients only. Attach the matching model.joblib as another notebook input and rerun.")
model = joblib.load(full_model_path)
if weights is not None:
    np.testing.assert_array_equal(weights, model["coefficients"])
    model["coefficients"] = weights
classes = [str(c) for c in model["classes"]]
assert set(classes) == set(COURSE_CODES) and len(set(COURSE_CODES.values())) == 41
codes = np.array([COURSE_CODES[c] for c in classes])
print(f"Loaded {model['coefficients'].shape} kernel coefficients from {MODEL_DIR.name} ({full_model_path.name})")

In [ ]:
def course_ids(stream):
    table = pd.read_csv(stream, dtype=str)
    if list(table) != ["id"] or len(table) != 1592 or not table.id.is_unique:
        return None
    ids = table.id.tolist()
    fingerprint = hashlib.sha256("\n".join(sorted(ids)).encode()).hexdigest()
    return ids if fingerprint == ID_HASH else None

def find_recordings():
    roots = [DATA] if DATA else [INPUT]
    for root in roots:
        if root.is_dir():
            for manifest in sorted(root.rglob("test.csv")):
                ids = course_ids(manifest)
                if ids is None:
                    continue
                for relative in ("audio/audio/test", "audio/test", "test"):
                    folder = manifest.parent / relative
                    if all((folder / (i+".wav")).is_file() for i in ids):
                        return ids, folder, None
        archives = [root] if root.is_file() else sorted(root.rglob("project4.zip"))
        for path in archives:
            with ZipFile(path) as archive:
                names = set(archive.namelist())
                for name in sorted(names):
                    if Path(name).name != "test.csv":
                        continue
                    ids = course_ids(BytesIO(archive.read(name)))
                    if ids is None:
                        continue
                    for relative in ("audio/audio/test", "audio/test", "test"):
                        folder = Path(name).parent / relative
                        if all((folder/(i+".wav")).as_posix() in names for i in ids):
                            return ids, folder, path
    raise FileNotFoundError("Attach the Project 4 test.csv and WAV directory (or project4.zip) under /kaggle/input.")

ids, folder, archive_path = find_recordings()
print(f"Found {len(ids)} Project 4 recordings")

## Acoustic features
Each WAV is converted to mono at 44.1 kHz. Extraction uses the same texture, pitch and covariance descriptors as training.

In [ ]:
RATE = 44100
FRAME = 2048
HOP = 882
BANDS = 72
FFT = 16384
BINS_PER_OCTAVE = 36
FIRST_MIDI = 26

@lru_cache(maxsize=4)
def filters(frame=FRAME, bands=BANDS, rate=RATE):
    frequencies = np.fft.rfftfreq(frame, 1 / rate)
    limits = np.log1p(np.array([30, min(20000, rate * .475)]) / 700)
    edges = 700 * np.expm1(np.linspace(limits[0], limits[1], bands + 2))
    rise = (frequencies - edges[:-2, None]) / np.diff(edges)[:-1, None]
    fall = (edges[2:, None] - frequencies) / np.diff(edges)[1:, None]
    mel = np.maximum(np.minimum(rise, fall), 0)
    mel /= np.maximum(mel.sum(axis=1, keepdims=True), 1e-12)
    window = np.hanning(frame).astype('float32')
    return csr_matrix(mel.astype('float32')), window

def summaries(values):
    return [values.mean(0), values.std(0), *np.percentile(values, [10, 50, 90], axis=0)]

def modulation(envelopes):
    """Windowed modulation spectra capture sustained pulses and repeated attacks."""
    width = 128
    if len(envelopes) < width:
        envelopes = np.pad(envelopes, ((0, width - len(envelopes)), (0, 0)), mode='edge')
    windows = np.lib.stride_tricks.sliding_window_view(envelopes, width, axis=0)[::width // 2]
    centered = (windows - windows.mean(axis=-1, keepdims=True)) * np.hanning(width).astype('float32')
    power = np.abs(rfft(centered, axis=-1)) ** 2
    edges = [1, 3, 6, 11, 19, 32, 49, 65]
    return [np.log1p(power[:, :, a:b].mean(axis=(0, 2))) for a, b in zip(edges, edges[1:])]

def periodicity(power, energy, rate=RATE):
    """Pool the strengths and pitch distribution of prominent autocorrelation peaks."""
    # Compression whitens the spectrum enough to expose overtones.
    correlation = irfft(power[::3] ** .65, axis=1)
    correlation /= np.maximum(correlation[:, :1], 1e-12)
    ranges = np.rint(np.array([28, 55, 110, 220, 440, 736]) * (rate / RATE)).astype(int)
    lags = np.arange(ranges[0], ranges[-1])
    middle = correlation[:, lags]
    local_peaks = (middle > correlation[:, lags - 1]) & (middle >= correlation[:, lags + 1])
    best = (middle * local_peaks).argmax(axis=1)
    clarity = np.maximum(middle[np.arange(len(middle)), best], 0)
    # Downweight near-silent frames. Weight stable periodic frames more strongly.
    loudness = energy[::3] / max(float(energy.max()), 1e-10)
    weights = clarity ** 2 * np.sqrt(loudness)
    midi = 69 + 12 * np.log2((rate / lags[best]) / 440)
    pitches = np.histogram(midi, bins=np.arange(33, 96, 2), weights=weights)[0]
    pitches /= max(weights.sum(), 1e-8)
    chroma = np.bincount(np.rint(midi).astype(int) % 12, weights=weights, minlength=12)
    chroma /= max(weights.sum(), 1e-8)
    octave_strength = np.stack([correlation[:, a:b].max(1) for a, b in zip(ranges, ranges[1:])], axis=1)
    return [pitches, chroma, *summaries(octave_strength), np.percentile(clarity, [10, 50, 90])]

def waveform_features(samples, rate=RATE):
    width, hop = round(FRAME * rate / RATE), round(HOP * rate / RATE)
    mel, window = filters(width, BANDS, rate)
    if len(samples) < width:
        samples = np.pad(samples, (0, width - len(samples)))
    frames = np.lib.stride_tricks.sliding_window_view(samples, width)[::hop]
    spectrum = rfft(frames * window, axis=1)
    power = spectrum.real ** 2 + spectrum.imag ** 2
    energy = power.sum(1)
    mel_energy = (mel @ power.T).T
    peak = max(float(mel_energy.max()), 1e-10)
    logmel = np.maximum(10 * np.log10(np.maximum(mel_energy / peak, 1e-9)), -80)
    # Each 20-ms frame has a log-mel spectral envelope and a cepstral description.
    features = [*summaries(logmel)]
    cepstra = dct(logmel, type=2, axis=1, norm='ortho')[:, :24]
    features.extend(summaries(cepstra))
    centered = cepstra - cepstra.mean(0)
    covariance = centered.T @ centered / len(centered)
    upper = covariance[np.triu_indices(covariance.shape[0], k=1)]
    features.append(np.sign(upper) * np.sqrt(np.abs(upper)))

    pooled = logmel.reshape(len(logmel), BANDS // 3, 3).mean(2)
    features.extend(modulation(pooled))
    # Haar differences across time, frequency and diagonals describe local contours.
    for step, bands in [(1, 1), (4, 2), (12, 3)]:
        smooth = uniform_filter1d(pooled, step, axis=0, mode='nearest')
        smooth = uniform_filter1d(smooth, bands, axis=1, mode='nearest')
        if len(smooth) <= step:
            smooth = np.pad(smooth, ((0, step + 1 - len(smooth)), (0, 0)), mode='edge')
        later, earlier = smooth[step:], smooth[:-step]
        for delta in [later - earlier, later[:, bands:] - later[:, :-bands],
                      later[:, bands:] - earlier[:, :-bands], later[:, :-bands] - earlier[:, bands:]]:
            features.extend([np.maximum(delta, 0).mean(0), np.maximum(-delta, 0).mean(0)])
    # Spectral flatness and prominence by octave separate tonal and noise-like sounds.
    octave_edges = [edge for edge in (1, 5, 10, 20, 40, 80, 160, 320, 640) if edge < power.shape[1]] + [power.shape[1]]
    flatness, prominence = [], []
    for low, high in zip(octave_edges, octave_edges[1:]):
        subband = np.maximum(power[:, low:high], 1e-9)
        arithmetic = np.log(subband.mean(1))
        flatness.append(np.log(subband).mean(1) - arithmetic)
        prominence.append(np.log(subband.max(1)) - arithmetic)
    features.extend(summaries(np.array(flatness + prominence).T))
    features.extend(periodicity(power, energy, rate))
    active = energy > max(float(energy.max()) * .01, 1e-10)
    positions = np.arange(len(energy)) / max(len(energy) - 1, 1)
    features.extend([np.array([np.log1p(len(samples) / rate), active.mean(),
                              positions[energy.argmax()], np.dot(positions, energy) / max(energy.sum(), 1e-9)])])
    result = np.concatenate(features).astype('float32')
    if not np.isfinite(result).all():
        raise ValueError('Non-finite texture descriptor')
    return result

In [ ]:
def log_covariance(rows):
    """Represent shrinkage covariance in log-Euclidean tangent coordinates."""
    centered = rows - rows.mean(0)
    covariance = centered.T @ centered / max(len(rows) - 1, 1)
    shrinkage = .08
    diagonal = max(covariance.trace() / len(covariance), 1.)
    covariance *= 1 - shrinkage
    covariance.flat[::len(covariance) + 1] += shrinkage * diagonal
    values, vectors = np.linalg.eigh(covariance)
    logged = (vectors * np.log(np.maximum(values, 1e-6))) @ vectors.T
    i, j = np.triu_indices(len(covariance))
    return logged[i, j] * np.where(i == j, 1., np.sqrt(2.))

def envelope_summary(energy, active):
    db = np.clip(10 * np.log10(np.maximum(energy, 1e-9)), -80., 0.)
    frames = db[active]
    # The proportion of each sound type and its evolving spectral shape provide
    # distinct cues for sustained notes, speech, machinery and short impacts.
    return [frames.mean(0), frames.std(0), *np.percentile(frames, [20, 80], axis=0),
            np.abs(np.diff(db, axis=0, prepend=db[:1])).mean(0)]

def resolution_features(audio, width, hop, bands, rate=44100):
    mel, window = filters(width, bands, rate)
    if len(audio) < width:
        audio = np.pad(audio, (0, width - len(audio)))
    frames = np.lib.stride_tricks.sliding_window_view(audio, width)[::hop]
    fft = rfft(frames * window, axis=1)
    power = fft.real ** 2 + fft.imag ** 2
    energy = power.sum(1)
    active = energy >= max(float(energy.max()) * .0003, 1e-10)
    if not active.any():
        active[:] = True
    mel_energy = (mel @ power.T).T
    mel_energy /= max(float(mel_energy.max()), 1e-10)
    db = np.maximum(10 * np.log10(np.maximum(mel_energy, 1e-9)), -80.)
    cepstra = np.ascontiguousarray(dct(db, type=2, axis=1, norm='ortho')[:, :20])
    # The zeroth coefficient sums all band energies. Scale it so shrinkage treats
    # the other cepstral dimensions and their derivatives more evenly.
    cepstra[:, 0] *= .25
    delta = correlate1d(cepstra, np.array([-2, -1, 0, 1, 2]) / 10., axis=0, mode='nearest')
    joint = np.column_stack((cepstra, 3. * delta))
    result = [log_covariance(joint[active]), joint[active].mean(0)]
    active_ids = np.flatnonzero(active)
    # Difference between onset and release distinguishes similar timbres.
    count = max(1, len(active_ids) // 4)
    result.append(cepstra[active_ids[:count]].mean(0) - cepstra[active_ids[-count:]].mean(0))
    # Soft median masks compare persistence across time with breadth across frequency.
    harmonic = median_filter(mel_energy, size=(11, 1), mode='nearest')
    percussive = median_filter(mel_energy, size=(1, 9), mode='nearest')
    harmonic **= 2
    percussive **= 2
    mask = harmonic / np.maximum(harmonic + percussive, 1e-12)
    result.extend(envelope_summary(mel_energy * mask, active))
    result.extend(envelope_summary(mel_energy * (1 - mask), active))
    result.extend([mask[active].mean(0), mask[active].std(0)])
    return result

def waveform_covariance(audio, rate=44100):
    # Short frames preserve impacts. Long frames resolve narrow, low-frequency harmonics.
    # Both describe continuous time on a fixed hop throughout each original recording.
    hop = round(882 * rate / 44100)
    result = resolution_features(audio, round(1024 * rate / 44100), hop, 48, rate)
    result.extend(resolution_features(audio, round(4096 * rate / 44100), hop, 72, rate))
    features = np.concatenate(result).astype('float32')
    if not np.isfinite(features).all():
        raise ValueError('Non-finite covariance descriptor')
    return features

In [ ]:
@lru_cache(maxsize=1)
def filterbank(rate=RATE):
    """Each log band spans two thirds of a semitone, centered a third of a semitone apart."""
    bottom = 440 * 2 ** ((FIRST_MIDI - 69) / 12)
    width = round(FFT * rate / RATE)
    count = int(np.floor(BINS_PER_OCTAVE * np.log2(min(19000, .47 * rate) / bottom)))
    centers = bottom * 2 ** (np.arange(count) / BINS_PER_OCTAVE)
    fft_hz = np.fft.rfftfreq(width, 1 / rate)
    indices, values, offsets = [], [], [0]
    for center in centers:
        low, high = center * 2 ** (-1 / BINS_PER_OCTAVE), center * 2 ** (1 / BINS_PER_OCTAVE)
        idx = np.where((fft_hz > low) & (fft_hz < high))[0]
        if len(idx) < 2:
            fbin = center * width / rate
            idx = np.array([int(fbin), int(fbin) + 1])
            weights = np.array([1 - fbin % 1, fbin % 1])
        else:
            weights = np.minimum((fft_hz[idx] - low) / (center - low),
                                 (high - fft_hz[idx]) / (high - center))
        indices.extend(idx)
        values.extend(weights / weights.sum())
        offsets.append(len(indices))
    bank = csr_matrix((np.array(values, dtype=np.float32), indices, offsets),
                      shape=(count, width // 2 + 1))
    window = np.hanning(width).astype(np.float32)
    return bank, window

@lru_cache(maxsize=1)
def harmonics():
    # Up to eight harmonics of fundamentals from MIDI 32 to 96, every third semitone.
    pitch = np.arange((32 - FIRST_MIDI) * 3, (96 - FIRST_MIDI) * 3 + 1)
    harmonics = np.arange(1, 9)
    idx = pitch[:, None] + np.rint(BINS_PER_OCTAVE * np.log2(harmonics)).astype(int)
    between = pitch[:, None] + np.rint(BINS_PER_OCTAVE * np.log2(harmonics + .5)).astype(int)
    weight = harmonics ** -.78
    return pitch, idx, between, (weight / weight.sum()).astype(np.float32)

def tonal_features(samples, rate=RATE):
    bank, window = filterbank(rate)
    # A fixed cap per clip bounds cost while spanning the entire recording.
    count = max(1, min(48, int(np.ceil(len(samples) / (2700 * rate / RATE)))))
    indices = np.linspace(0, len(samples) - 1, count).astype(int)
    padded = np.pad(samples, (len(window) // 2, len(window) // 2))
    frames = np.lib.stride_tricks.sliding_window_view(padded, len(window))[indices]
    spectrum = rfft(frames * window)
    power = np.square(spectrum.real) + np.square(spectrum.imag)
    level = power.sum(1)
    active = level >= max(float(level.max()) * .003, 1e-12)
    if not active.any():
        active[:] = True
    mel = (bank @ power[active].T).T
    floor = max(float(mel.max()) * 1e-9, 1.e-12)
    log_band = 10 * np.log10(np.maximum(mel, floor))
    log_band -= np.max(log_band, axis=1, keepdims=True)
    envelope = gaussian_filter1d(log_band, 1.6, axis=1, mode="nearest")
    features = [envelope[:, 1::3].mean(0), envelope[:, 1::3].std(0),
                np.percentile(envelope[:, 1::3], 85, axis=0)]
    # Whiten the broad spectral envelope before comparing harmonic combs.
    white = log_band - gaussian_filter1d(log_band, 6, axis=1, mode="nearest")
    pitch, on, off, weight = harmonics()
    # For lower-rate material, renormalize harmonic templates below Nyquist.
    if off.max() >= log_band.shape[1]:
        weights = weight * (off < log_band.shape[1])
        weights /= np.maximum(weights.sum(1, keepdims=True), 1e-8)
        on = on.clip(max=log_band.shape[1]-1)
        off = off.clip(max=log_band.shape[1]-1)
        salience = np.sum((.70 * white[:, on] + .30 * (log_band[:, on] - log_band[:, off])) * weights, axis=2)
    else:
        salience = ((.70 * white[:, on] + .30 * (log_band[:, on] - log_band[:, off])) @ weight)
    # An inaudible fundamental is plausible, but penalize combs with negligible low harmonics.
    audibility = log_band[:, on[:, :3]].max(axis=2)
    salience += .24 * np.minimum(audibility + 24, 0)
    pitch_scores = gaussian_filter1d(salience, 1.1, axis=1)[:, ::3]
    features += [pitch_scores.mean(0), pitch_scores.std(0), np.percentile(pitch_scores, 90, axis=0)]
    best = np.argmax(salience, axis=1)
    row = np.arange(len(best))
    confidence = np.maximum(salience[row, best], .01)
    mass = confidence / confidence.sum()
    histogram = np.bincount(np.rint(best / 3).astype(int), weights=mass, minlength=pitch_scores.shape[1])
    # Smooth the pitch histogram to avoid hard boundaries between neighbouring notes.
    features.append(gaussian_filter1d(histogram, 1.5))
    # Aligned overtones distinguish instruments that share a fundamental frequency.
    aligned = log_band[row[:, None], on[best]]
    aligned -= aligned.max(1, keepdims=True)
    contrast = white[row[:, None], on[best]]
    odd_even = aligned[:, 0::2] - aligned[:, 1::2]
    aligned_features = np.column_stack((aligned, contrast, odd_even))
    features += [np.average(aligned_features, axis=0, weights=mass), aligned_features.std(0),
                 *np.percentile(aligned_features, [15, 85], axis=0)]
    features += [aligned_features[:max(1, len(best)//3)].mean(0),
                 aligned_features[-max(1, len(best)//3):].mean(0)]
    prominence = np.maximum(white, 0)
    octave_width = BINS_PER_OCTAVE
    for lo in range(0, prominence.shape[1], octave_width):
        a = prominence[:, lo:lo + octave_width]
        features.append(np.array([a.mean(), a.std(), a.max(1).mean(), np.mean(a > 4)]))
    movement = np.abs(np.diff(best, prepend=best[0])) / 3
    features.append(np.array([np.average(best / 3 + 32, weights=mass), np.std(best / 3),
                              movement.mean(), np.mean(movement > 2),
                              np.mean(confidence), np.std(confidence), np.log1p(len(samples) / rate)]))
    return np.concatenate(features).astype(np.float32)

In [ ]:
VIEWS = {"texture": waveform_features, "tonal": tonal_features, "covariance": waveform_covariance}
VIEW_WEIGHTS = {"texture": .45, "tonal": .20, "covariance": .35}

def waveform_views(source):
    samples,rate = sf.read(source,dtype="float32",always_2d=True)
    samples = samples.mean(axis=1)
    if rate!=44100:
        factor=gcd(44100,rate)
        samples=resample_poly(samples,44100//factor,rate//factor)
    if not len(samples) or not np.isfinite(samples).all():
        raise ValueError("Expected a nonempty audio file with finite samples")
    return {name:fn(samples)[None,:] for name,fn in VIEWS.items()}

In [ ]:
def predict_batch(features):
    blocks = []
    for name, scaler in model["scalers"].items():
        x = np.vstack([row[name] for row in features])
        x = scaler.transform(x).clip(-5, 5)
        blocks.append(normalize(x, copy=False) * np.float32(np.sqrt(VIEW_WEIGHTS[name])))
    query = np.column_stack(blocks)
    sim = (np.minimum(query @ model["reference"].T, 1.) - 1.) * model["kernel_scale"]
    scores = (.5*np.exp(2*sim) + .5*np.exp(5*sim)) @ model["coefficients"]
    if not np.isfinite(scores).all():
        raise ValueError("Nonfinite model scores")
    return codes[np.argsort(-scores, axis=1, kind="stable")[:, :3]]

from contextlib import nullcontext

ranked = []
with (ZipFile(archive_path) if archive_path else nullcontext()) as archive:
    for start in range(0, len(ids), 64):
        paths = [folder/(i+".wav") for i in ids[start:start+64]]
        sources = [BytesIO(archive.read(p.as_posix())) for p in paths] if archive else paths
        features = [waveform_views(source) for source in sources]
        ranked.extend(" ".join(row) for row in predict_batch(features))
        if len(ranked) % 256 == 0 or len(ranked) == len(ids):
            print(f"Scored {len(ranked)}/{len(ids)}")

submission = pd.DataFrame({"id": ids, "label": ranked})
assert all(len(s.split()) == len(set(s.split())) == 3 and set(s.split()) <= set(codes) for s in ranked)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT, index=False)
assert pd.read_csv(OUTPUT, dtype=str).equals(submission)
print(OUTPUT)
print(submission.head().to_string(index=False))